# Projeto Forecast — 06. Exportação para o Power BI

Este notebook junta as saídas das etapas anteriores e salva tudo em **Parquet** na pasta `data/powerbi/`, já no formato de **modelo estrela** que o Power BI usa melhor:

| Arquivo | Tipo | Grão (uma linha por) |
|---|---|---|
| `dim_pais.parquet` | Dimensão | país |
| `dim_ano.parquet` | Dimensão | ano (1970–2030) |
| `fato_populacao.parquet` | Fato | país × ano, histórico (1970–2022) e previsão (2023–2027 e 2030) |
| `fato_idosos.parquet` | Fato | país × ano, população com 65+ do Banco Mundial (1970–2024) |
| `fato_crescimento_idosos.parquet` | Fato | país, comparação 2000 → 2022 |

As tabelas fato ficam no formato **longo** (uma linha por país e ano). Assim o Power BI desenha uma linha contínua do passado até 2030 e aplica os filtros de continente e de país em todas as visualizações.

Por que Parquet e não CSV: o arquivo guarda os tipos das colunas (inteiro, decimal, texto), é menor e carrega mais rápido. O Power BI lê com **Obter dados → Parquet**.

Como executar: rode os notebooks 01 a 05 antes deste.

In [1]:
import pandas as pd
from pathlib import Path

DATA = Path('data')
SAIDA = DATA / 'powerbi'
SAIDA.mkdir(exist_ok=True)

base = pd.read_csv(DATA / 'world_population_limpo.csv')
forecast_anual = pd.read_csv(DATA / 'forecast_anual_2022_2027.csv')
forecast_2030 = pd.read_csv(DATA / 'forecast_populacao.csv')
idosos = pd.read_csv(DATA / 'idosos_65mais_banco_mundial.csv')
crescimento = pd.read_csv(DATA / 'crescimento_e_idosos_2000_2022.csv')

for nome, df in [('base limpa', base), ('forecast anual', forecast_anual), ('forecast 2030', forecast_2030),
                 ('idosos', idosos), ('crescimento x idosos', crescimento)]:
    print(f'{nome:22} {df.shape}')

base limpa             (234, 17)
forecast anual         (234, 13)
forecast 2030          (234, 7)
idosos                 (14355, 5)
crescimento x idosos   (214, 12)


## 1. Dimensão de países

Uma linha por país, com a chave `cca3` (código ISO de 3 letras) que liga todas as tabelas fato. Os atributos descritivos (nome, capital, continente, área) ficam só aqui, e não repetidos em cada fato.

In [2]:
dim_pais = (base[['cca3', 'pais', 'capital', 'continente', 'area_km2', 'densidade_km2', 'rank', 'pct_pop_mundial']]
            .rename(columns={'rank': 'rank_pop_2022', 'pct_pop_mundial': 'pct_pop_mundial_2022'})
            .sort_values('pais')
            .reset_index(drop=True))

assert dim_pais['cca3'].is_unique
dim_pais.head()

,cca3,pais,capital,continente,area_km2,densidade_km2,rank_pop_2022,pct_pop_mundial_2022
0,AFG,Afghanistan,Kabul,Asia,652230,63.0587,36,0.52
1,ALB,Albania,Tirana,Europe,28748,98.8702,138,0.04
2,DZA,Algeria,Algiers,Africa,2381741,18.8531,34,0.56
3,ASM,American Samoa,Pago Pago,Oceania,199,222.4774,213,0.00
4,AND,Andorra,Andorra la Vella,Europe,468,170.5641,203,0.00


## 2. Fato de população (histórico + previsão)

Junta em uma única coluna `populacao`:

- **Histórico**: 1970, 1980, 1990, 2000, 2010, 2015, 2020 e 2022, da base limpa;
- **Previsão**: 2023 a 2027 (`forecast_anual_2022_2027.csv`) e 2030 (`forecast_populacao.csv`).

A coluna `tipo` separa o real do projetado, para usar como legenda no Power BI.

In [3]:
colunas_hist = [c for c in base.columns if c.startswith('pop_')]
historico = (base.melt(id_vars='cca3', value_vars=colunas_hist, var_name='ano', value_name='populacao')
             .assign(ano=lambda d: d['ano'].str.removeprefix('pop_').astype(int), tipo='Histórico'))

# o forecast anual só tem o nome do país; o código vem da dimensão
anual = forecast_anual.merge(dim_pais[['pais', 'cca3']], on='pais', how='left')
assert anual['cca3'].notna().all(), 'país do forecast anual sem código cca3'
previsao_anual = (anual.melt(id_vars='cca3', value_vars=[f'pop_{a}' for a in range(2023, 2028)],
                             var_name='ano', value_name='populacao')
                  .assign(ano=lambda d: d['ano'].str.removeprefix('pop_').astype(int)))
previsao_2030 = forecast_2030[['cca3', 'pop_2030']].rename(columns={'pop_2030': 'populacao'}).assign(ano=2030)
previsao = pd.concat([previsao_anual, previsao_2030]).assign(tipo='Previsão')

fato_populacao = (pd.concat([historico, previsao])
                  [['cca3', 'ano', 'tipo', 'populacao']]
                  .astype({'ano': 'int64', 'populacao': 'int64'})
                  .sort_values(['cca3', 'ano'])
                  .reset_index(drop=True))

print(fato_populacao.shape)
fato_populacao.groupby(['ano', 'tipo'])['populacao'].sum().div(1e9).round(3).rename('mundo (bi)').to_frame()

(3276, 4)


,,mundo (bi)
ano,tipo,
1970,Histórico,3.694
1980,Histórico,4.442
1990,Histórico,5.314
2000,Histórico,6.147
2010,Histórico,6.984
2015,Histórico,7.425
2020,Histórico,7.839
2022,Histórico,7.973
2023,Previsão,8.052


## 3. Fato de idosos (Banco Mundial)

O arquivo do Banco Mundial também traz agregados (Mundo, regiões, faixas de renda). Eles são **retirados** aqui, porque não existem na dimensão de países e somariam em dobro no Power BI; o total mundial passa a ser a soma dos países.

Também calculamos `populacao_bm` (população total segundo o Banco Mundial = idosos ÷ % de idosos). Com ela o Power BI calcula o **% de idosos de um continente ou do mundo** de forma ponderada (soma dos idosos ÷ soma da população), em vez de fazer a média simples dos percentuais dos países.

In [4]:
fato_idosos = (idosos[idosos['cca3'].isin(dim_pais['cca3'])]
               .drop(columns='nome_bm')
               .dropna(subset=['idosos_65mais', 'pct_idosos_65mais'])
               .assign(populacao_bm=lambda d: (d['idosos_65mais'] / d['pct_idosos_65mais'] * 100).round())
               .astype({'ano': 'int64', 'idosos_65mais': 'int64', 'populacao_bm': 'int64'})
               [['cca3', 'ano', 'idosos_65mais', 'pct_idosos_65mais', 'populacao_bm']]
               .sort_values(['cca3', 'ano'])
               .reset_index(drop=True))

print(fato_idosos.shape, '| países:', fato_idosos['cca3'].nunique(),
      '| anos:', fato_idosos['ano'].min(), '–', fato_idosos['ano'].max())
fato_idosos.head()

(11750, 5) | países: 214 | anos: 1970 – 2024


,cca3,ano,idosos_65mais,pct_idosos_65mais,populacao_bm
0,ABW,1970,2373,4.025411,58951
1,ABW,1971,2507,4.264169,58792
2,ABW,1972,2614,4.503740,58041
3,ABW,1973,2760,4.733703,58305
4,ABW,1974,2892,4.956588,58347


## 4. Fato de crescimento x idosos (2000 → 2022)

A tabela do `05_idosos.ipynb`, sem as colunas de nome e continente (que já estão em `dim_pais`). Serve para os rankings e para o gráfico de dispersão "crescimento da população × variação da fatia de idosos".

In [5]:
fato_crescimento_idosos = (crescimento.drop(columns=['pais', 'continente'])
                           .rename(columns={'cresc_pop_%': 'cresc_pop_pct', 'cresc_idosos_%': 'cresc_idosos_pct'})
                           .astype({'idosos_2000': 'Int64', 'idosos_2022': 'Int64'}))
fato_crescimento_idosos.head()

,cca3,pop_2000,pop_2022,idosos_2000,idosos_2022,pct_idosos_2000,pct_idosos_2022,cresc_pop_pct,cresc_idosos_pct,var_pct_idosos_pp
0,AFG,19542982,41128771,455342,955689,2.261970,2.355142,110.452893,109.883780,0.093172
1,ALB,3182021,2842321,226447,386778,7.330691,15.776338,-10.675605,70.802881,8.445647
2,DZA,30774621,44903225,1354555,2813669,4.383121,6.186964,45.909920,107.719066,1.803843
3,ASM,58230,44273,1864,3434,3.277665,7.103627,-23.968745,84.227468,3.825962
4,AND,66097,79824,8094,11931,12.321687,14.968948,20.767962,47.405486,2.647261


## 5. Dimensão de anos

Uma linha por ano de 1970 a 2030, ligada às duas fatos anuais. Ela permite usar **um único filtro de ano** para população e idosos ao mesmo tempo. A coluna `decada` serve para agrupar nos gráficos.

In [6]:
anos = sorted(set(fato_populacao['ano']) | set(fato_idosos['ano']))
dim_ano = pd.DataFrame({'ano': range(min(anos), max(anos) + 1)})
dim_ano['decada'] = (dim_ano['ano'] // 10 * 10).astype(str) + 's'
dim_ano['periodo'] = dim_ano['ano'].map(lambda a: 'Previsão' if a > 2022 else 'Histórico')
dim_ano.tail()

,ano,decada,periodo
56,2026,2020s,Previsão
57,2027,2020s,Previsão
58,2028,2020s,Previsão
59,2029,2020s,Previsão
60,2030,2030s,Previsão


## 6. Verificações

Antes de salvar, conferimos as regras que o modelo do Power BI pressupõe:

- chaves únicas nas dimensões e nas fatos (sem linha repetida para o mesmo país e ano);
- todo `cca3` e todo `ano` das fatos existem nas dimensões (relacionamentos sem órfãos);
- totais mundiais iguais aos das etapas anteriores (8,60 bi em 2030).

In [7]:
assert dim_ano['ano'].is_unique
assert not fato_populacao.duplicated(['cca3', 'ano']).any()
assert not fato_idosos.duplicated(['cca3', 'ano']).any()
assert fato_crescimento_idosos['cca3'].is_unique
for nome, fato in [('populacao', fato_populacao), ('idosos', fato_idosos), ('crescimento', fato_crescimento_idosos)]:
    assert fato['cca3'].isin(dim_pais['cca3']).all(), f'cca3 órfão em {nome}'
for fato in [fato_populacao, fato_idosos]:
    assert fato['ano'].isin(dim_ano['ano']).all()

mundo = fato_populacao.groupby('ano')['populacao'].sum()
assert mundo[2022] == base['pop_2022'].sum()
assert 8.3e9 < mundo[2030] < 8.9e9
print(f'Mundo: 2022 = {mundo[2022] / 1e9:.3f} bi | 2030 = {mundo[2030] / 1e9:.3f} bi')
print('Todas as verificações passaram.')

Mundo: 2022 = 7.973 bi | 2030 = 8.600 bi
Todas as verificações passaram.


## 7. Exportação

In [8]:
tabelas = {
    'dim_pais': dim_pais,
    'dim_ano': dim_ano,
    'fato_populacao': fato_populacao,
    'fato_idosos': fato_idosos,
    'fato_crescimento_idosos': fato_crescimento_idosos,
}
for nome, df in tabelas.items():
    caminho = SAIDA / f'{nome}.parquet'
    df.to_parquet(caminho, index=False)
    print(f'Salvo: {caminho} {df.shape}  ({caminho.stat().st_size / 1024:.0f} KB)')

Salvo: data\powerbi\dim_pais.parquet (234, 8)  (17 KB)
Salvo: data\powerbi\dim_ano.parquet (61, 3)  (3 KB)
Salvo: data\powerbi\fato_populacao.parquet (3276, 4)  (27 KB)
Salvo: data\powerbi\fato_idosos.parquet (11750, 5)  (277 KB)
Salvo: data\powerbi\fato_crescimento_idosos.parquet (214, 10)  (23 KB)


## Conclusão

- Os resultados do projeto ficam em 5 arquivos Parquet em `data/powerbi/`, organizados em modelo estrela: 2 dimensões (`dim_pais`, `dim_ano`) e 3 fatos.
- A população histórica e a prevista estão em uma única tabela, separadas pela coluna `tipo`. Assim um só gráfico de linha mostra de 1970 a 2030.
- Os agregados do Banco Mundial foram retirados, para não contar o mundo em dobro. O % de idosos de grupos de países deve ser calculado por medida ponderada, e não pela média dos percentuais.
- O passo a passo para montar o dashboard (relacionamentos, medidas DAX e páginas) está em [`POWERBI.md`](POWERBI.md).